# 02b — JAC IEV40 Processing

This notebook processes the JAC IEV40 telemetry dataset.

## Source
- **File**: `dataset/archive/dataset.csv`
- **Vehicle**: JAC IEV40 electric vehicle
- **Output**: `data/interim/jac/jac_standardized.parquet`

## Critical Semantic Rules
- **AIR**: Sensor flag (values 0/2 only) — NOT ambient temperature
- **VOL**: Raw values (0-379) — NOT verified battery voltage
- **CUR**: Raw current — NOT assumed HV battery current
- **BRK/ACC**: Raw sensor signals — NOT percentages

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import json
from src.data.jac_parser import discover_jac_file, process_jac, generate_jac_cleaning_report

## Step 1 — Discover the JAC source file

In [ ]:
file_info = discover_jac_file(base_path='../dataset/archive')
print(f"File: {file_info['filename']}")
print(f"Size: {file_info['file_size_mb']} MB")
print(f"Rows: {file_info['rows']:,}")
print(f"Columns ({file_info['columns']}): {file_info['column_names']}")

## Step 2 — Process and standardize

In [ ]:
summary = process_jac(
    base_path='../dataset/archive',
    output_dir='../data/interim/jac'
)
generate_jac_cleaning_report(summary, output_path='../docs/jac_cleaning_report.md')

## Step 3 — Inspect the standardized output

In [ ]:
df = pd.read_parquet('../data/interim/jac/jac_standardized.parquet')
print(f"Shape: {df.shape}")
print(f"\nColumns:\n{list(df.columns)}")
df.head(10)

In [ ]:
# Basic statistics
df.describe()

## Step 4 — Quality flags overview

In [ ]:
qf_cols = [c for c in df.columns if c.startswith('quality_')]
for col in qf_cols:
    valid = (df[col] == 1).sum()
    invalid = (df[col] == 0).sum()
    print(f"{col}: valid={valid:,}, flagged={invalid:,}, pct_valid={100*valid/len(df):.1f}%")

## Step 5 — Timestamp analysis

In [ ]:
valid_ts = df[df['quality_timestamp'] == 1]['timestamp']
print(f"Valid timestamps: {len(valid_ts):,} / {len(df):,}")
print(f"Earliest: {valid_ts.min()}")
print(f"Latest: {valid_ts.max()}")
print(f"Duplicates: {valid_ts.duplicated().sum()}")

# Sampling intervals
sorted_ts = valid_ts.sort_values()
diffs = sorted_ts.diff().dropna().dt.total_seconds()
print(f"\nSampling interval (seconds):")
print(f"  Min:    {diffs.min():.1f}")
print(f"  Median: {diffs.median():.1f}")
print(f"  Mean:   {diffs.mean():.1f}")
print(f"  Max:    {diffs.max():.1f}")
print(f"  Gaps > 60s: {(diffs > 60).sum()}")

## Step 6 — Sensor guardrails verification

Confirm that no forbidden standardized columns were created.

In [ ]:
forbidden = ['ambient_temperature_c', 'battery_voltage_v', 'battery_current_a',
             'battery_power_kw', 'energy_consumption_kwh_per_km', 'regen_energy_kwh']
for col in forbidden:
    status = 'PRESENT (ERROR!)' if col in df.columns else 'Absent (correct)'
    print(f"  {col}: {status}")

# Verify raw columns exist
raw_expected = ['vol_raw', 'current_raw', 'air_sensor_flag', 'brake_raw', 'accelerator_raw']
for col in raw_expected:
    status = 'Present (correct)' if col in df.columns else 'MISSING (ERROR!)'
    print(f"  {col}: {status}")

## Step 7 — Source lineage check

In [ ]:
assert (df['source_dataset'] == 'JAC_IEV40').all(), 'source_dataset mismatch'
assert (df['source_file'] == 'dataset.csv').all(), 'source_file mismatch'
assert df['source_row_id'].is_monotonic_increasing, 'source_row_id not monotonic'
print('Source lineage fields: OK')

## Step 8 — Processing summary

In [ ]:
with open('../data/interim/jac/processing_summary.json', 'r') as f:
    summary = json.load(f)

for k, v in summary.items():
    if isinstance(v, dict):
        print(f"\n{k}:")
        for kk, vv in v.items():
            print(f"  {kk}: {vv}")
    elif isinstance(v, list) and len(v) > 5:
        print(f"{k}: [{len(v)} items]")
    else:
        print(f"{k}: {v}")